# Indoor Scene Change Detection — Training & Evaluation Pipeline


## 1. Environment Setup


In [ ]:
!nvidia-smi


In [ ]:
!pip install ultralytics pandas numpy scipy scikit-learn matplotlib seaborn opencv-python -q
import os, json, shutil, glob, zipfile, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from PIL import Image as PILImage
from sklearn.metrics import classification_report, confusion_matrix
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO
from google.colab import files
from IPython.display import Image, display

print("Libraries ready.")


## 8. Change Detection Features (Add / Delete / Move)


In [ ]:
def compute_iou(box1, box2):
    x1, y1, x2, y2 = box1
    x3, y3, x4, y4 = box2
    xi1, yi1 = max(x1, x3), max(y1, y3)
    xi2, yi2 = min(x2, x4), min(y2, y4)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    area1 = (x2 - x1) * (y2 - y1)
    area2 = (x4 - x3) * (y4 - y3)
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

def extract_boxes(results, model):
    boxes = []
    if results[0].boxes is not None:
        for box in results[0].boxes:
            xyxy = box.xyxy[0].cpu().numpy().tolist()
            cls_id = int(box.cls[0].cpu().numpy())
            boxes.append({
                'class': model.names[cls_id],
                'bbox': xyxy,
                'confidence': float(box.conf[0].cpu().numpy())
            })
    return boxes

def normalize_name(s):
    return s.replace('-', ' ').replace('_', ' ').strip().lower()

def warp_bbox(bbox, H):

    if H is None:
        return bbox
    x1, y1, x2, y2 = bbox
    corners = np.array([[[x1, y1]], [[x2, y1]], [[x2, y2]], [[x1, y2]]], dtype=np.float32)
    warped = cv2.perspectiveTransform(corners, H).reshape(-1, 2)
    wx1, wy1 = float(warped[:, 0].min()), float(warped[:, 1].min())
    wx2, wy2 = float(warped[:, 0].max()), float(warped[:, 1].max())
    return [wx1, wy1, wx2, wy2]

def get_image_size(path):

    with PILImage.open(path) as im:
        return im.size

def crop_bbox(img, bbox, pad=4):

    if img is None:
        return None
    h, w = img.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
    x2, y2 = min(w, x2 + pad), min(h, y2 + pad)
    if x2 <= x1 or y2 <= y1:
        return None
    return img[y1:y2, x1:x2]

def appearance_similarity(crop_a, crop_b):

    if crop_a is None or crop_b is None or crop_a.size == 0 or crop_b.size == 0:
        return 0.0
    hsv_a = cv2.cvtColor(crop_a, cv2.COLOR_BGR2HSV)
    hsv_b = cv2.cvtColor(crop_b, cv2.COLOR_BGR2HSV)
    hist_a = cv2.calcHist([hsv_a], [0, 1], None, [50, 60], [0, 180, 0, 256])
    hist_b = cv2.calcHist([hsv_b], [0, 1], None, [50, 60], [0, 180, 0, 256])
    cv2.normalize(hist_a, hist_a)
    cv2.normalize(hist_b, hist_b)
    corr = cv2.compareHist(hist_a, hist_b, cv2.HISTCMP_CORREL)
    return float(max(0.0, corr))

def match_boxes(before_boxes, after_boxes, H=None, move_thresh=50,
                 image_diag=None, max_match_dist_frac=0.35,
                 before_img=None, after_img=None, min_appearance_sim=0.35):

    events = []
    matched_pairs = []
    max_match_dist = (max_match_dist_frac * image_diag) if image_diag else float('inf')
    use_appearance = before_img is not None and after_img is not None

    classes = set(b['class'] for b in before_boxes) | set(a['class'] for a in after_boxes)

    for cls in classes:
        b_idx = [i for i, b in enumerate(before_boxes) if b['class'] == cls]
        a_idx = [j for j, a in enumerate(after_boxes) if a['class'] == cls]
        if not b_idx and not a_idx:
            continue

        b_centers = []
        for i in b_idx:
            wb = warp_bbox(before_boxes[i]['bbox'], H)
            b_centers.append(((wb[0] + wb[2]) / 2, (wb[1] + wb[3]) / 2))
        a_centers = []
        for j in a_idx:
            ab = after_boxes[j]['bbox']
            a_centers.append(((ab[0] + ab[2]) / 2, (ab[1] + ab[3]) / 2))

        matched_b, matched_a = set(), set()

        if b_idx and a_idx:
            cost = np.zeros((len(b_idx), len(a_idx)))
            for bi, (bx, by) in enumerate(b_centers):
                for aj, (ax, ay) in enumerate(a_centers):
                    cost[bi, aj] = np.hypot(ax - bx, ay - by)
            row_ind, col_ind = linear_sum_assignment(cost)

            for bi, aj in zip(row_ind, col_ind):
                dist = float(cost[bi, aj])
                if dist > max_match_dist:
                    continue

                i, j = b_idx[bi], a_idx[aj]
                b, a = before_boxes[i], after_boxes[j]

                sim = 1.0
                if use_appearance:
                    crop_b_img = crop_bbox(before_img, b['bbox'])
                    crop_a_img = crop_bbox(after_img, a['bbox'])
                    sim = appearance_similarity(crop_b_img, crop_a_img)
                    if sim < min_appearance_sim:

                        continue

                matched_b.add(bi)
                matched_a.add(aj)
                matched_pairs.append((b, a, dist, sim))
                if dist > move_thresh:
                    events.append({'type': 'Move', 'object': cls,
                                    'confidence': (b['confidence'] + a['confidence']) / 2,
                                    'movement': dist, 'appearance_sim': sim})

        for bi, i in enumerate(b_idx):
            if bi not in matched_b:
                events.append({'type': 'Delete', 'object': cls,
                                'confidence': before_boxes[i]['confidence'], 'movement': 0.0})
        for aj, j in enumerate(a_idx):
            if aj not in matched_a:
                events.append({'type': 'Add', 'object': cls,
                                'confidence': after_boxes[j]['confidence'], 'movement': 0.0})

    return events, matched_pairs

def summarize_events(events, matched_pairs=None):

    matched_pairs = matched_pairs or []
    add_events = [e for e in events if e['type'] == 'Add']
    del_events = [e for e in events if e['type'] == 'Delete']
    move_events = [e for e in events if e['type'] == 'Move']

    def best(evs):
        return max(evs, key=lambda e: e['confidence']) if evs else None

    best_add, best_del, best_move = best(add_events), best(del_events), best(move_events)
    max_matched_dist = max((d for _, _, d, _ in matched_pairs), default=0.0)
    avg_appearance_sim = float(np.mean([s for _, _, _, s in matched_pairs])) if matched_pairs else 0.0

    return {
        'n_add': len(add_events), 'n_del': len(del_events), 'n_move': len(move_events),
        'net': len(add_events) - len(del_events),
        'best_add_conf': best_add['confidence'] if best_add else 0.0,
        'best_del_conf': best_del['confidence'] if best_del else 0.0,
        'best_move_conf': best_move['confidence'] if best_move else 0.0,
        'best_move_dist': best_move['movement'] if best_move else 0.0,
        'max_matched_dist': max_matched_dist,
        'avg_appearance_sim': avg_appearance_sim,
        'best_add': best_add, 'best_del': best_del, 'best_move': best_move,
    }

def object_for_type(predicted_type, stats, matched_pairs):

    key = {'Add': 'best_add', 'Delete': 'best_del', 'Move': 'best_move'}.get(predicted_type)
    cand = stats.get(key) if key else None
    if cand:
        return cand['object']
    if matched_pairs:
        b, a, dist, sim = max(matched_pairs, key=lambda t: t[2])
        return b['class']
    return 'Unknown'

def build_feature_dict(before_boxes, after_boxes, ious, residual_px, move_thresh, events, matched_pairs):

    stats = summarize_events(events, matched_pairs)
    best_move = stats['best_move']
    best_move_appearance_sim = best_move['appearance_sim'] if best_move else 0.0

    n_matched = len(matched_pairs)
    frac_matched_before = n_matched / len(before_boxes) if before_boxes else 0.0
    frac_matched_after = n_matched / len(after_boxes) if after_boxes else 0.0

    n_classes_changed = len({e['object'] for e in events})

    add_del_conf_margin = stats['best_add_conf'] - stats['best_del_conf']
    move_conf_margin = stats['best_move_conf'] - max(stats['best_add_conf'], stats['best_del_conf'])

    area_ratios = []
    for b, a, dist, sim in matched_pairs:
        ab, aa = b['bbox'], a['bbox']
        area_b = max(1e-6, (ab[2] - ab[0]) * (ab[3] - ab[1]))
        area_a = max(1e-6, (aa[2] - aa[0]) * (aa[3] - aa[1]))
        area_ratios.append(min(area_a, area_b) / max(area_a, area_b))
    avg_matched_area_ratio = float(np.mean(area_ratios)) if area_ratios else 1.0

    net_count_diff = len(after_boxes) - len(before_boxes)
    total_events = stats['n_add'] + stats['n_del'] + stats['n_move']
    move_event_ratio = (stats['n_move'] / total_events) if total_events else 0.0
    is_net_ambiguous = 1.0 if abs(net_count_diff) <= 1 else 0.0

    feat = {
        'before_count': len(before_boxes), 'after_count': len(after_boxes),
        'net_count_diff': net_count_diff,
        'total_objects': len(before_boxes) + len(after_boxes),
        'max_iou': max(ious) if ious else 0.0, 'avg_iou': float(np.mean(ious)) if ious else 0.0,
        'camera_shift_residual_px': residual_px, 'move_thresh_used': move_thresh,
        'n_add': stats['n_add'], 'n_del': stats['n_del'], 'n_move': stats['n_move'],
        'best_add_conf': stats['best_add_conf'], 'best_del_conf': stats['best_del_conf'],
        'best_move_conf': stats['best_move_conf'], 'best_move_dist': stats['best_move_dist'],
        'max_matched_dist': stats['max_matched_dist'], 'avg_appearance_sim': stats['avg_appearance_sim'],
        'best_move_appearance_sim': best_move_appearance_sim,
        'n_matched': n_matched, 'frac_matched_before': frac_matched_before,
        'frac_matched_after': frac_matched_after,
        'add_del_conf_margin': add_del_conf_margin, 'move_conf_margin': move_conf_margin,
        'n_classes_changed': n_classes_changed, 'avg_matched_area_ratio': avg_matched_area_ratio,
        'total_events': total_events, 'move_event_ratio': move_event_ratio,
        'is_net_ambiguous': is_net_ambiguous,
    }
    return feat, stats

print("Change detection feature-extraction functions ready (Add / Delete / Move only, "
      "camera-shift aware, Hungarian-matched, appearance-gated). The Add/Delete/Move "
      "decision itself is made by the ML classifier in Section 10.")


### 8a. Compensate for Camera Angle Shift Between Before/After Shots


In [ ]:
import cv2

def estimate_camera_shift(before_path, after_path, max_features=2000, good_match_pct=0.2,
                           ransac_reproj_thresh=5.0):

    img1 = cv2.imread(before_path, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(after_path, cv2.IMREAD_GRAYSCALE)
    if img1 is None or img2 is None:
        return np.eye(3), 0.0

    orb = cv2.ORB_create(max_features)
    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)
    if des1 is None or des2 is None or len(kp1) < 10 or len(kp2) < 10:
        return np.eye(3), 0.0

    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    matches = matcher.match(des1, des2)
    matches = sorted(matches, key=lambda m: m.distance)
    n_good = max(15, int(len(matches) * good_match_pct))
    matches = matches[:n_good]
    if len(matches) < 8:
        return np.eye(3), 0.0

    pts1 = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    pts2 = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    H, inlier_mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, ransacReprojThreshold=ransac_reproj_thresh)
    if H is None:
        return np.eye(3), 0.0

    inliers = inlier_mask.ravel().astype(bool)
    if inliers.sum() < 8:
        return np.eye(3), 0.0

    pts1_proj = cv2.perspectiveTransform(pts1[inliers], H)
    residuals = np.linalg.norm((pts1_proj - pts2[inliers]).reshape(-1, 2), axis=1)
    residual_px = float(np.median(residuals))

    return H, residual_px

def adaptive_move_thresh(residual_px, min_thresh=15.0, safety_factor=3.0):

    return max(min_thresh, safety_factor * residual_px)

print("Camera-shift compensation ready: estimate_camera_shift(), adaptive_move_thresh().")


## 9. Filter to Add/Delete/Move Pairs (safety-net re-check)


In [ ]:
CHANGE_LABEL_MAP = {'add': 'Add', 'delete': 'Delete', 'move': 'Move'}
KEEP_CHANGE_TYPES = set(CHANGE_LABEL_MAP.keys())

pairs_df = ALL_PAIRS_DF[ALL_PAIRS_DF['change_type'].isin(KEEP_CHANGE_TYPES)].copy()
pairs_df['true_change'] = pairs_df['change_type'].map(CHANGE_LABEL_MAP)
pairs_df = pairs_df.rename(columns={'object': 'true_object'})

n_dropped = len(ALL_PAIRS_DF) - len(pairs_df)
if n_dropped:
    print(f"WARNING: {n_dropped} pairs in ALL_PAIRS_DF had change_type outside "
          f"{sorted(KEEP_CHANGE_TYPES)} -- this shouldn't happen given Section 3b's "
          f"filename filter; investigate if it's non-zero.")
else:
    print("Confirmed: ALL_PAIRS_DF was already Add/Delete/Move only (Section 3b filter working as expected).")

print(f"Built {len(pairs_df)} valid Add/Delete/Move pairs")
print(pairs_df['split'].value_counts())
pairs_df[['pair_id', 'before_path', 'after_path', 'true_change', 'true_object', 'room', 'split']].head()


In [ ]:

_winner_row = comparison_df.loc[comparison_df['f1'].idxmax()]
_winner = _winner_row['model']
ACTIVE_MODEL_PATH = YOLO_WEIGHTS if _winner == 'YOLO11s' else RTDETR_WEIGHTS
print(f"Detector selected by validation F1: {_winner} (f1={_winner_row['f1']:.3f}) -> using {ACTIVE_MODEL_PATH}")

CONF_THRESH = 0.4

MOVE_MIN_THRESH = 15.0
MOVE_SAFETY_FACTOR = 3.0
MAX_MATCH_DIST_FRAC = 0.5
MIN_APPEARANCE_SIM = 0.35
active_model = YOLO(ACTIVE_MODEL_PATH)

detection_cache = []
for _, row in pairs_df.iterrows():
    before_res = active_model(row['before_path'], conf=CONF_THRESH, verbose=False)
    after_res  = active_model(row['after_path'], conf=CONF_THRESH, verbose=False)
    before_boxes = extract_boxes(before_res, active_model)
    after_boxes  = extract_boxes(after_res, active_model)

    H, residual_px = estimate_camera_shift(row['before_path'], row['after_path'])
    after_w, after_h = get_image_size(row['after_path'])
    image_diag = float(np.hypot(after_w, after_h))

    detection_cache.append({
        'pair_id': row['pair_id'], 'before_boxes': before_boxes, 'after_boxes': after_boxes,
        'H': H, 'residual_px': residual_px, 'image_diag': image_diag,
        'before_path': row['before_path'], 'after_path': row['after_path'],
        'true_change': row['true_change'], 'true_object': row['true_object'], 'room': row['room'],
        'split': row['split'],
    })

print(f"Cached detections + camera-shift alignment for {len(detection_cache)} pairs.")

def build_results_df(cache, max_match_dist_frac, min_appearance_sim,
                      move_min_thresh=MOVE_MIN_THRESH, move_safety_factor=MOVE_SAFETY_FACTOR):

    rows = []
    contexts = []
    for c in cache:
        move_thresh = adaptive_move_thresh(c['residual_px'], move_min_thresh, move_safety_factor)
        before_img = cv2.imread(c['before_path'])
        after_img  = cv2.imread(c['after_path'])

        events, matched_pairs = match_boxes(
            c['before_boxes'], c['after_boxes'], H=c['H'], move_thresh=move_thresh,
            image_diag=c['image_diag'], max_match_dist_frac=max_match_dist_frac,
            before_img=before_img, after_img=after_img, min_appearance_sim=min_appearance_sim)

        ious = [compute_iou(warp_bbox(b['bbox'], c['H']), a['bbox'])
                for b, a, dist, sim in matched_pairs]

        feat, stats = build_feature_dict(c['before_boxes'], c['after_boxes'], ious,
                                          c['residual_px'], move_thresh, events, matched_pairs)
        rows.append({
            'pair_id': c['pair_id'], **feat,
            'true_change': c['true_change'], 'true_object': c['true_object'], 'room': c['room'],
            'split': c['split'],
        })
        contexts.append({'stats': stats, 'matched_pairs': matched_pairs})
    return pd.DataFrame(rows), contexts

results_df, pair_contexts = build_results_df(detection_cache, MAX_MATCH_DIST_FRAC, MIN_APPEARANCE_SIM)
results_df


In [ ]:
print("True change distribution by split:")
print(results_df.groupby('split')['true_change'].value_counts().unstack(fill_value=0))
print()
print("No predictions exist yet at this point -- Section 9b tunes the matching thresholds")
print("(train-fit / validation-scored) and Section 10 trains + tests the classifier.")


## 9b. Tune Matching Thresholds (parallelized grid sweep, cached detections)


In [ ]:
import itertools
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier as _QuickRF
from sklearn.metrics import f1_score as _quick_f1

def _quick_ml_val_f1(df):
    exclude = {'pair_id', 'true_change', 'true_object', 'room', 'split'}
    feat_cols = [c for c in df.columns if c not in exclude]
    tr = df[df['split'] == 'train']
    va = df[df['split'] == 'validation']
    quick_clf = _QuickRF(n_estimators=150, max_depth=6, class_weight='balanced', random_state=42, n_jobs=1)
    quick_clf.fit(tr[feat_cols].fillna(0.0), tr['true_change'])
    val_pred = quick_clf.predict(va[feat_cols].fillna(0.0))
    return _quick_f1(va['true_change'], val_pred, average='macro')

def _eval_combo(frac, sim):
    df_try, _ = build_results_df(detection_cache, frac, sim)
    f1 = _quick_ml_val_f1(df_try)
    return {'max_match_dist_frac': frac, 'min_appearance_sim': sim, 'quick_val_macro_f1': f1}

frac_grid = [0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7]
sim_grid = [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.55]

sweep_rows = Parallel(n_jobs=-1, prefer='processes')(
    delayed(_eval_combo)(frac, sim) for frac, sim in itertools.product(frac_grid, sim_grid)
)

sweep_df = pd.DataFrame(sweep_rows).sort_values('quick_val_macro_f1', ascending=False).reset_index(drop=True)
print(sweep_df.head(10).to_string(index=False))

best = sweep_df.iloc[0]
print(f"\nBest combo: max_match_dist_frac={best['max_match_dist_frac']}, "
      f"min_appearance_sim={best['min_appearance_sim']}  ->  quick validation macro-F1 {best['quick_val_macro_f1']:.3f}")

MAX_MATCH_DIST_FRAC = float(best['max_match_dist_frac'])
MIN_APPEARANCE_SIM = float(best['min_appearance_sim'])
results_df, pair_contexts = build_results_df(detection_cache, MAX_MATCH_DIST_FRAC, MIN_APPEARANCE_SIM)
results_df
